<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-13/notebooks/ClimatePipeline/01_ClimateDataDownloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateDataDownloader

Descargador genérico de observaciones climáticas IDEAM publicadas mediante Socrata. Guarda datos crudos en Parquet particionado por variable, fuente, departamento, año y mes.

Este notebook **solo descarga y normaliza tipos básicos**. No suma, promedia ni audita las observaciones, porque esas reglas dependen de cada variable climática.

## 1. Configuración

### Uso rápido

1. Escoja un `DATASET_ID` y un `VARIABLE_NOMBRE`.
2. Configure uno o ambos departamentos permitidos.
3. Configure listas de años y meses. Para todos los meses use `list(range(1, 13))`; para enero a abril use `[1, 2, 3, 4]`.
4. Mantenga `SOBRESCRIBIR_PARQUET = False` para reanudar descargas existentes.
5. Cambie `EJECUTAR_DESCARGA = True` y ejecute el notebook completo.

Fuentes candidatas:

| Variable | Dataset ID |
|---|---|
| Temperatura ambiente | `sbwg-7ju4` |
| Temperatura mínima | `afdg-3zpb` |
| Temperatura máxima | `ccvq-rp9s` |
| Humedad del aire | `uext-mhny` |
| Precipitación | `s54a-sgyg` |
| Velocidad del viento | `sgfv-3yp8` |
| Presión atmosférica | `62tk-nxj5` |

In [1]:
from pathlib import Path

import pandas as pd
import requests

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Fuente a descargar. Cambiar ambos valores juntos.
DATASET_ID = '62tk-nxj5'
VARIABLE_NOMBRE = 'presion_atmosferica'

# Es obligatorio indicar uno o ambos departamentos del alcance.
DESCARGA_DEPARTAMENTOS = [
    'CUNDINAMARCA',
]

# Ejemplos: [2025], [2024, 2025] o list(range(2019, 2026)).
DESCARGA_ANIOS = [2024]

# Ejemplos: [1, 2, 3, 4] o list(range(1, 13)).
DESCARGA_MESES = list(range(1, 13))

DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = None
SOBRESCRIBIR_PARQUET = False
REQUEST_TIMEOUT = 180
REQUEST_REINTENTOS = 3
APP_TOKEN = None  # Opcional. No subir tokens reales al repositorio.

# Banderita de seguridad para Run all.
EJECUTAR_DESCARGA = True

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)

print({
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': DESCARGA_DEPARTAMENTOS,
    'anios': DESCARGA_ANIOS,
    'meses': DESCARGA_MESES,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
    'sobrescribir': SOBRESCRIBIR_PARQUET,
    'ejecutar': EJECUTAR_DESCARGA,
    'processed_root': str(PROCESSED_ROOT),
})


Mounted at /content/drive
{'dataset_id': '62tk-nxj5', 'variable': 'presion_atmosferica', 'departamentos': ['CUNDINAMARCA'], 'anios': [2024], 'meses': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'limit': 1000, 'max_lotes': None, 'sobrescribir': False, 'ejecutar': True, 'processed_root': '/content/drive/MyDrive/eco2026_processed'}


## 2. Validación y plan de descarga

La configuración se valida antes de crear carpetas o consultar datos. Los departamentos se restringen al alcance aprobado: Boyacá y Cundinamarca.

In [2]:
import re
import unicodedata

DATASET_ID_PATTERN = re.compile(r'^[a-z0-9]{4}-[a-z0-9]{4}$', re.IGNORECASE)
DEPARTAMENTOS_PERMITIDOS = {'BOYACÁ', 'CUNDINAMARCA'}


def slugificar(valor):
    """Normaliza etiquetas para construir rutas deterministas."""
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    if not texto:
        raise ValueError(f'No se pudo construir una etiqueta de ruta para {valor!r}.')
    return texto


def unicos_ordenados(valores):
    return sorted(set(valores))


def validar_configuracion():
    if not DATASET_ID_PATTERN.fullmatch(str(DATASET_ID)):
        raise ValueError('DATASET_ID debe tener el formato xxxx-xxxx.')
    if not str(VARIABLE_NOMBRE).strip():
        raise ValueError('VARIABLE_NOMBRE es obligatorio.')
    if not DESCARGA_DEPARTAMENTOS:
        raise ValueError('Debe configurar al menos un departamento.')

    departamentos = unicos_ordenados(str(d).strip().upper() for d in DESCARGA_DEPARTAMENTOS)
    no_permitidos = set(departamentos) - DEPARTAMENTOS_PERMITIDOS
    if no_permitidos:
        raise ValueError(
            f'Departamentos fuera del alcance: {sorted(no_permitidos)}. '
            f'Permitidos: {sorted(DEPARTAMENTOS_PERMITIDOS)}.'
        )

    if not DESCARGA_ANIOS:
        raise ValueError('DESCARGA_ANIOS no puede estar vacío.')
    anios = unicos_ordenados(int(a) for a in DESCARGA_ANIOS)
    if any(a < 1900 or a > 2100 for a in anios):
        raise ValueError(f'Años fuera de rango: {anios}.')

    if not DESCARGA_MESES:
        raise ValueError('DESCARGA_MESES no puede estar vacío.')
    meses = unicos_ordenados(int(m) for m in DESCARGA_MESES)
    if any(m < 1 or m > 12 for m in meses):
        raise ValueError(f'Los meses deben estar entre 1 y 12: {meses}.')

    if int(DESCARGA_LIMIT) <= 0:
        raise ValueError('DESCARGA_LIMIT debe ser positivo.')
    if DESCARGA_MAX_LOTES is not None and int(DESCARGA_MAX_LOTES) <= 0:
        raise ValueError('DESCARGA_MAX_LOTES debe ser None o un entero positivo.')

    return departamentos, anios, meses


def construir_plan(departamentos, anios, meses):
    return [
        {'departamento': departamento, 'anio': anio, 'mes': mes}
        for departamento in departamentos
        for anio in anios
        for mes in meses
    ]

## 3. Acceso a Socrata

Las consultas usan `LIMIT` y `OFFSET` dentro de `$query`. Cada solicitud tiene reintentos para errores transitorios y mantiene un orden estable por fecha, estación, sensor e ID interno.

In [3]:
import time


def headers_socrata():
    headers = {'Accept': 'application/json', 'User-Agent': 'RAIZ-ClimateDataDownloader/1.0'}
    if APP_TOKEN:
        headers['X-App-Token'] = APP_TOKEN
    return headers


def consultar_metadata(dataset_id):
    url = f'https://www.datos.gov.co/api/views/{dataset_id}'
    response = requests.get(
        url,
        headers=headers_socrata(),
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def validar_esquema_ideam(metadata):
    campos = {columna.get('fieldName') for columna in metadata.get('columns', [])}
    obligatorios = {'departamento', 'fechaobservacion'}
    faltantes = obligatorios - campos
    if faltantes:
        raise ValueError(
            f'El dataset no tiene el esquema climático esperado. Faltan: {sorted(faltantes)}.'
        )
    return campos


def consultar_lote(dataset_id, where, limit, offset, order, timeout=REQUEST_TIMEOUT):
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'
    query = (
        f'SELECT * WHERE {where} ORDER BY {order} '
        f'LIMIT {int(limit)} OFFSET {int(offset)}'
    )

    ultimo_error = None
    for intento in range(1, REQUEST_REINTENTOS + 1):
        try:
            response = requests.get(
                url,
                params={'$query': query},
                headers=headers_socrata(),
                timeout=timeout,
            )
            response.raise_for_status()
            return pd.DataFrame(response.json())
        except requests.RequestException as exc:
            ultimo_error = exc
            if intento == REQUEST_REINTENTOS:
                break
            espera = min(2 ** (intento - 1), 30)
            print(
                f'Consulta falló en intento {intento}/{REQUEST_REINTENTOS}: {exc}. '
                f'Reintentando en {espera} s...'
            )
            time.sleep(espera)

    raise ultimo_error


def construir_orden(campos):
    candidatos = ['fechaobservacion', 'codigoestacion', 'codigosensor']
    presentes = [campo for campo in candidatos if campo in campos]
    presentes.append(':id')
    return ', '.join(presentes)

## 4. Particiones y reanudación

Cada lote de hasta 1.000 filas se guarda como un archivo `part-xxxxx.parquet`. Todos los archivos de un mismo departamento, año y mes quedan en la misma carpeta.

### `SOBRESCRIBIR_PARQUET = False`

Es el modo recomendado. Si existen partes consecutivas, la descarga empieza en el siguiente índice y usa el `OFFSET` correspondiente. Si la partición ya estaba completa, la API devolverá cero filas y no se escribirá nada nuevo.

Antes de reanudar se revisa el número de filas de cada archivo. Todas las partes salvo la última deben tener exactamente `DESCARGA_LIMIT` filas. Si el último archivo tiene menos, la partición se considera completa. Esto también evita reanudar accidentalmente con un `LIMIT` diferente al usado en la primera corrida.

### `SOBRESCRIBIR_PARQUET = True`

La descarga vuelve a empezar en el lote cero y reemplaza archivos con el mismo nombre. No crea deliberadamente otra carpeta. Tampoco elimina partes antiguas sobrantes; por eso no se recomienda salvo que se entienda el estado de la partición.

`mkdir(..., exist_ok=True)` reutiliza la ruta montada. Si Google Drive muestra dos carpetas visualmente iguales, suele indicar rutas raíz distintas, accesos directos duplicados o diferencias invisibles en el nombre; no es efecto directo de esta bandera.

In [4]:
from datetime import datetime

PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')


def inicio_mes_siguiente(anio, mes):
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1


def ruta_particion(variable, dataset_id, departamento, anio, mes):
    return (
        PROCESSED_ROOT
        / 'clima_crudo'
        / f'variable={slugificar(variable)}'
        / f'fuente={dataset_id.lower()}'
        / f'departamento={slugificar(departamento)}'
        / f'anio={int(anio)}'
        / f'mes={int(mes):02d}'
    )


def partes_existentes(output_dir):
    encontrados = []
    for archivo in output_dir.glob('part-*.parquet'):
        coincidencia = PART_PATTERN.fullmatch(archivo.name)
        if coincidencia:
            encontrados.append((int(coincidencia.group(1)), archivo))
    encontrados.sort(key=lambda item: item[0])
    return encontrados


def validar_secuencia_partes(partes):
    indices = [indice for indice, _ in partes]
    if not indices:
        return
    esperados = list(range(indices[-1] + 1))
    if indices != esperados:
        faltantes = sorted(set(esperados) - set(indices))
        raise RuntimeError(
            f'La partición tiene huecos en sus archivos: {faltantes}. '
            'No se reanuda para evitar saltar registros.'
        )


def validar_filas_partes(partes, limit):
    if not partes:
        return [], False

    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('Se necesita pyarrow para validar los Parquet existentes.') from exc

    filas = [pq.ParquetFile(archivo).metadata.num_rows for _, archivo in partes]
    inconsistentes = [
        indice
        for (indice, _), filas_archivo in zip(partes[:-1], filas[:-1])
        if filas_archivo != int(limit)
    ]
    if inconsistentes:
        raise RuntimeError(
            f'Las partes {inconsistentes} no tienen DESCARGA_LIMIT={limit} filas. '
            'No es seguro calcular el OFFSET para reanudar.'
        )
    if filas[-1] > int(limit):
        raise RuntimeError(
            f'La última parte tiene {filas[-1]} filas, más que el limit={limit}.'
        )

    particion_completa = filas[-1] < int(limit)
    return filas, particion_completa


def normalizar_lote(df, dataset_id, variable):
    df = df.copy()
    if 'fechaobservacion' in df.columns:
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'], errors='coerce')
    for columna in ['valorobservado', 'latitud', 'longitud']:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')
    df['dataset_id'] = dataset_id
    df['variable_fuente'] = slugificar(variable)
    return df


def descargar_particion(
    dataset_id,
    variable,
    departamento,
    anio,
    mes,
    campos,
    limit=1000,
    max_lotes=None,
    sobrescribir=False,
):
    output_dir = ruta_particion(variable, dataset_id, departamento, anio, mes)
    output_dir.mkdir(parents=True, exist_ok=True)

    existentes = partes_existentes(output_dir)
    validar_secuencia_partes(existentes)
    filas_existentes, ya_completa = validar_filas_partes(existentes, limit)
    inicio_idx = 0 if sobrescribir else len(existentes)
    lote_idx = inicio_idx
    offset = lote_idx * int(limit)
    lotes_consultados = 0
    filas_nuevas = 0
    detalle = []
    inicio_tiempo = time.perf_counter()

    anio_fin, mes_fin = inicio_mes_siguiente(anio, mes)
    fecha_inicio = f'{int(anio):04d}-{int(mes):02d}-01T00:00:00'
    fecha_fin = f'{anio_fin:04d}-{mes_fin:02d}-01T00:00:00'
    departamento_sql = str(departamento).replace("'", "''")
    where = (
        f"departamento = '{departamento_sql}' "
        f"AND fechaobservacion >= '{fecha_inicio}' "
        f"AND fechaobservacion < '{fecha_fin}'"
    )
    order = construir_orden(campos)

    print(f'Carpeta: {output_dir}')
    print(f'Partes existentes: {len(existentes):,}')
    if filas_existentes:
        print(f'Filas por parte existente: {filas_existentes[:10]}')
    print(f'Inicio lote={lote_idx:,} | offset={offset:,} | sobrescribir={sobrescribir}')

    if ya_completa and not sobrescribir:
        print('La última parte tiene menos filas que el limit; la partición ya está completa.')
        resumen = {
            'dataset_id': dataset_id,
            'variable': slugificar(variable),
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'estado': 'ya_completa',
            'partes_existentes_inicio': len(existentes),
            'partes_escritas_corrida': 0,
            'filas_corrida': 0,
            'duracion_segundos': round(time.perf_counter() - inicio_tiempo, 2),
            'carpeta': str(output_dir),
        }
        return resumen, pd.DataFrame()

    estado_particion = 'completa'
    ultimo_lote_escrito = None

    while True:
        if max_lotes is not None and lotes_consultados >= max_lotes:
            estado_particion = 'pausada_por_max_lotes'
            print(f'Corte por max_lotes={max_lotes}.')
            break

        reloj_inicio = datetime.now()
        lote_tiempo = time.perf_counter()
        print(
            f'Consultando lote {lote_idx:,} | offset={offset:,} | '
            f'limit={int(limit):,} | inicio={reloj_inicio:%H:%M:%S}'
        )
        df_lote = consultar_lote(
            dataset_id=dataset_id,
            where=where,
            limit=limit,
            offset=offset,
            order=order,
        )
        lotes_consultados += 1

        if df_lote.empty:
            print('La API no devolvió más filas para esta partición.')
            break

        df_lote = normalizar_lote(df_lote, dataset_id, variable)
        output_path = output_dir / f'part-{lote_idx:05d}.parquet'
        ya_existia = output_path.exists()
        df_lote.to_parquet(output_path, index=False)

        filas_lote = len(df_lote)
        filas_nuevas += filas_lote
        ultimo_lote_escrito = lote_idx
        reloj_fin = datetime.now()
        duracion_lote = round(time.perf_counter() - lote_tiempo, 2)
        print(
            f'Guardado {output_path.name} | filas={filas_lote:,} | '
            f'duración={duracion_lote:.2f} s | fin={reloj_fin:%H:%M:%S}'
        )

        detalle.append({
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'lote': lote_idx,
            'offset': offset,
            'filas': filas_lote,
            'estado': 'sobrescrito' if ya_existia else 'escrito',
            'inicio': reloj_inicio.isoformat(timespec='seconds'),
            'fin': reloj_fin.isoformat(timespec='seconds'),
            'duracion_segundos': duracion_lote,
            'archivo': str(output_path),
        })

        if filas_lote < limit:
            print(f'Último lote detectado con {filas_lote:,} filas.')
            break

        lote_idx += 1
        offset += int(limit)

    if sobrescribir and existentes and ultimo_lote_escrito is not None:
        sobrantes = [indice for indice, _ in existentes if indice > ultimo_lote_escrito]
        if sobrantes:
            print(
                'ADVERTENCIA: quedaron partes antiguas con índices mayores al último '
                f'lote escrito: {sobrantes[:10]}.'
            )

    resumen = {
        'dataset_id': dataset_id,
        'variable': slugificar(variable),
        'departamento': departamento,
        'anio': int(anio),
        'mes': int(mes),
        'estado': estado_particion,
        'partes_existentes_inicio': len(existentes),
        'partes_escritas_corrida': len(detalle),
        'filas_corrida': filas_nuevas,
        'duracion_segundos': round(time.perf_counter() - inicio_tiempo, 2),
        'carpeta': str(output_dir),
    }
    return resumen, pd.DataFrame(detalle)

## 5. Ejecución

El plan recorre todas las combinaciones configuradas. Un error en una partición queda registrado y no borra las partes descargadas anteriormente.

In [5]:
resumen_particiones = pd.DataFrame()
detalle_lotes = pd.DataFrame()

departamentos, anios, meses = validar_configuracion()
plan_descarga = construir_plan(departamentos, anios, meses)
vista_plan = pd.DataFrame(plan_descarga)
vista_plan['carpeta'] = vista_plan.apply(
    lambda fila: str(
        ruta_particion(
            VARIABLE_NOMBRE,
            DATASET_ID,
            fila['departamento'],
            fila['anio'],
            fila['mes'],
        )
    ),
    axis=1,
)
display(Markdown(f'## Plan configurado — {len(plan_descarga)} particiones'))
display(vista_plan)

if not EJECUTAR_DESCARGA:
    print('Descarga desactivada. Revise la configuración y cambie EJECUTAR_DESCARGA a True.')
else:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError('Para guardar Parquet instale pyarrow: !pip install pyarrow') from exc

    metadata = consultar_metadata(DATASET_ID)
    campos_dataset = validar_esquema_ideam(metadata)

    print(f"Dataset: {metadata.get('name')} ({DATASET_ID})")
    print(f'Variable de salida: {slugificar(VARIABLE_NOMBRE)}')
    print(f'Particiones a procesar: {len(plan_descarga):,}')
    print(f'Ruta raíz: {PROCESSED_ROOT / "clima_crudo"}')

    resumenes = []
    detalles = []

    for indice, item in enumerate(plan_descarga, start=1):
        titulo = (
            f"➡️ Partición {indice}/{len(plan_descarga)} — "
            f"{item['departamento']} | {item['anio']}-{item['mes']:02d}"
        )
        display(Markdown(f'---\n\n## {titulo}'))

        try:
            resumen, detalle = descargar_particion(
                dataset_id=DATASET_ID,
                variable=VARIABLE_NOMBRE,
                departamento=item['departamento'],
                anio=item['anio'],
                mes=item['mes'],
                campos=campos_dataset,
                limit=DESCARGA_LIMIT,
                max_lotes=DESCARGA_MAX_LOTES,
                sobrescribir=SOBRESCRIBIR_PARQUET,
            )
            resumenes.append(resumen)
            if not detalle.empty:
                detalles.append(detalle)
        except Exception as exc:
            print(f'ERROR en partición: {type(exc).__name__}: {exc}')
            resumenes.append({
                'dataset_id': DATASET_ID,
                'variable': slugificar(VARIABLE_NOMBRE),
                **item,
                'estado': 'error',
                'error': f'{type(exc).__name__}: {exc}',
            })

    resumen_particiones = pd.DataFrame(resumenes)
    detalle_lotes = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()

    display(Markdown('---\n\n## Resumen final'))
    display(resumen_particiones)
    display(detalle_lotes)

## Plan configurado — 12 particiones

,departamento,anio,mes,carpeta
0,CUNDINAMARCA,2024,1,/content/drive/MyDrive/eco2026_processed/clima...
1,CUNDINAMARCA,2024,2,/content/drive/MyDrive/eco2026_processed/clima...
2,CUNDINAMARCA,2024,3,/content/drive/MyDrive/eco2026_processed/clima...
3,CUNDINAMARCA,2024,4,/content/drive/MyDrive/eco2026_processed/clima...
4,CUNDINAMARCA,2024,5,/content/drive/MyDrive/eco2026_processed/clima...
5,CUNDINAMARCA,2024,6,/content/drive/MyDrive/eco2026_processed/clima...
6,CUNDINAMARCA,2024,7,/content/drive/MyDrive/eco2026_processed/clima...
7,CUNDINAMARCA,2024,8,/content/drive/MyDrive/eco2026_processed/clima...
8,CUNDINAMARCA,2024,9,/content/drive/MyDrive/eco2026_processed/clima...
9,CUNDINAMARCA,2024,10,/content/drive/MyDrive/eco2026_processed/clima...


Dataset: Presión Atmosférica (62tk-nxj5)
Variable de salida: presion_atmosferica
Particiones a procesar: 12
Ruta raíz: /content/drive/MyDrive/eco2026_processed/clima_crudo


---

## ➡️ Partición 1/12 — CUNDINAMARCA | 2024-01

Carpeta: /content/drive/MyDrive/eco2026_processed/clima_crudo/variable=presion_atmosferica/fuente=62tk-nxj5/departamento=cundinamarca/anio=2024/mes=01
Partes existentes: 0
Inicio lote=0 | offset=0 | sobrescribir=False
Consultando lote 0 | offset=0 | limit=1,000 | inicio=04:11:31
Guardado part-00000.parquet | filas=1,000 | duración=0.62 s | fin=04:11:31
Consultando lote 1 | offset=1,000 | limit=1,000 | inicio=04:11:31
Guardado part-00001.parquet | filas=1,000 | duración=0.55 s | fin=04:11:32
Consultando lote 2 | offset=2,000 | limit=1,000 | inicio=04:11:32
Guardado part-00002.parquet | filas=1,000 | duración=0.53 s | fin=04:11:32
Consultando lote 3 | offset=3,000 | limit=1,000 | inicio=04:11:32
Guardado part-00003.parquet | filas=1,000 | duración=0.62 s | fin=04:11:33
Consultando lote 4 | offset=4,000 | limit=1,000 | inicio=04:11:33
Guardado part-00004.parquet | filas=1,000 | duración=0.47 s | fin=04:11:34
Consultando lote 5 | offset=5,000 | limit=1,000 | inicio=04:11:34
Guardado part-0

...

---

## ➡️ Partición 12/12 — CUNDINAMARCA | 2024-12

Carpeta: /content/drive/MyDrive/eco2026_processed/clima_crudo/variable=presion_atmosferica/fuente=62tk-nxj5/departamento=cundinamarca/anio=2024/mes=12
Partes existentes: 0
Inicio lote=0 | offset=0 | sobrescribir=False
Consultando lote 0 | offset=0 | limit=1,000 | inicio=04:16:00
Guardado part-00000.parquet | filas=1,000 | duración=0.55 s | fin=04:16:00
Consultando lote 1 | offset=1,000 | limit=1,000 | inicio=04:16:00
Guardado part-00001.parquet | filas=1,000 | duración=0.59 s | fin=04:16:01
Consultando lote 2 | offset=2,000 | limit=1,000 | inicio=04:16:01
Guardado part-00002.parquet | filas=1,000 | duración=0.59 s | fin=04:16:02
Consultando lote 3 | offset=3,000 | limit=1,000 | inicio=04:16:02
Guardado part-00003.parquet | filas=1,000 | duración=0.58 s | fin=04:16:02
Consultando lote 4 | offset=4,000 | limit=1,000 | inicio=04:16:02
Guardado part-00004.parquet | filas=1,000 | duración=0.75 s | fin=04:16:03
Consultando lote 5 | offset=5,000 | limit=1,000 | inicio=04:16:03
Guardado part-0

---

## Resumen final

,dataset_id,variable,departamento,anio,mes,estado,partes_existentes_inicio,partes_escritas_corrida,filas_corrida,duracion_segundos,carpeta
0,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,1,completa,0,36,35811,22.17,/content/drive/MyDrive/eco2026_processed/clima...
1,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,2,completa,0,22,21924,13.95,/content/drive/MyDrive/eco2026_processed/clima...
2,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,3,completa,0,20,19670,11.62,/content/drive/MyDrive/eco2026_processed/clima...
3,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,4,completa,0,16,15830,12.99,/content/drive/MyDrive/eco2026_processed/clima...
4,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,5,completa,0,20,19410,12.64,/content/drive/MyDrive/eco2026_processed/clima...
5,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,6,completa,0,22,21983,13.48,/content/drive/MyDrive/eco2026_processed/clima...
6,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,7,completa,0,24,23966,16.45,/content/drive/MyDrive/eco2026_processed/clima...
7,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,8,completa,0,43,42574,28.53,/content/drive/MyDrive/eco2026_processed/clima...
8,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,9,completa,0,33,32053,20.84,/content/drive/MyDrive/eco2026_processed/clima...
9,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,10,completa,0,66,65417,50.66,/content/drive/MyDrive/eco2026_processed/clima...


,departamento,anio,mes,lote,offset,filas,estado,inicio,fin,duracion_segundos,archivo
0,CUNDINAMARCA,2024,1,0,0,1000,escrito,2026-07-13T04:11:31,2026-07-13T04:11:31,0.62,/content/drive/MyDrive/eco2026_processed/clima...
1,CUNDINAMARCA,2024,1,1,1000,1000,escrito,2026-07-13T04:11:31,2026-07-13T04:11:32,0.55,/content/drive/MyDrive/eco2026_processed/clima...
2,CUNDINAMARCA,2024,1,2,2000,1000,escrito,2026-07-13T04:11:32,2026-07-13T04:11:32,0.53,/content/drive/MyDrive/eco2026_processed/clima...
3,CUNDINAMARCA,2024,1,3,3000,1000,escrito,2026-07-13T04:11:32,2026-07-13T04:11:33,0.62,/content/drive/MyDrive/eco2026_processed/clima...
4,CUNDINAMARCA,2024,1,4,4000,1000,escrito,2026-07-13T04:11:33,2026-07-13T04:11:34,0.47,/content/drive/MyDrive/eco2026_processed/clima...
...,...,...,...,...,...,...,...,...,...,...,...
442,CUNDINAMARCA,2024,12,68,68000,1000,escrito,2026-07-13T04:17:00,2026-07-13T04:17:01,1.10,/content/drive/MyDrive/eco2026_processed/clima...
443,CUNDINAMARCA,2024,12,69,69000,1000,escrito,2026-07-13T04:17:01,2026-07-13T04:17:03,1.41,/content/drive/MyDrive/eco2026_processed/clima...
444,CUNDINAMARCA,2024,12,70,70000,1000,escrito,2026-07-13T04:17:03,2026-07-13T04:17:04,1.24,/content/drive/MyDrive/eco2026_processed/clima...
445,CUNDINAMARCA,2024,12,71,71000,1000,escrito,2026-07-13T04:17:04,2026-07-13T04:17:05,1.28,/content/drive/MyDrive/eco2026_processed/clima...


In [6]:
from pathlib import Path

def obtener_tamano_carpeta(ruta_carpeta):
    total_bytes = 0
    for archivo in ruta_carpeta.glob('*.parquet'):
        total_bytes += archivo.stat().st_size
    return total_bytes

# Filtrar el resumen de particiones para el año 2024
resumen_2024 = resumen_particiones[resumen_particiones['anio'] == 2024]

total_tamano_bytes = 0
for index, row in resumen_2024.iterrows():
    carpeta_path = Path(row['carpeta'])
    total_tamano_bytes += obtener_tamano_carpeta(carpeta_path)

# Convertir a un formato legible (MB, GB)
total_tamano_mb = total_tamano_bytes / (1024 * 1024)
total_tamano_gb = total_tamano_bytes / (1024 * 1024 * 1024)

print(f"El tamaño total de la descarga para el año 2024 es:")
print(f"- {total_tamano_bytes:,} bytes")
print(f"- {total_tamano_mb:.2f} MB")
print(f"- {total_tamano_gb:.2f} GB")

El tamaño total de la descarga para el año 2024 es:
- 8,242,920 bytes
- 7.86 MB
- 0.01 GB


## 6. Qué queda pendiente

La salida conserva observaciones crudas normalizadas y trazabilidad de la fuente. Las siguientes tareas deben desarrollarse por variable y fuera de este notebook:

- Revisar duplicados por estación, sensor y fecha.
- Medir frecuencia y cobertura temporal por estación.
- Definir reglas de valores físicamente plausibles.
- Construir agregados diarios y por periodo agrícola.
- Decidir cómo combinar estaciones dentro de un municipio.

No use una suma genérica para temperatura, humedad, presión o viento.